In [51]:
import pandas as pd
import numpy as np
from gensim.models import KeyedVectors
import ast

In [52]:
def convertir_a_string(valor):
    if pd.isna(valor):
        return ""
    
    if isinstance(valor, str):
        try:
            valor_parseado = ast.literal_eval(valor)
            
            if isinstance(valor_parseado, list):
                return " ".join(str(x) for x in valor_parseado)
            
            return str(valor_parseado)
        
        except (ValueError, SyntaxError):
            return valor
    
    if isinstance(valor, list):
        return " ".join(str(x) for x in valor)
    
    return str(valor)


In [53]:
columnas = [
    "titulo",
    "autores",
    "generos",
    "serie",
    "sinopsis"
]

for columna in columnas:
    df[columna] = df[columna].apply(convertir_a_string)


CAMBIAR LA VARIABLE DE PURO!!


In [54]:
df["puro"] = df[columnas].agg(" ".join, axis=1)

df["puro"] = (
    df["puro"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [55]:
df[["titulo", "autores", "generos", "serie", "sinopsis", "puro"]].head()


,titulo,autores,generos,serie,sinopsis,puro
0,El papá de las bellezas,Felipe Trigo,Realista Relato,,"El papá de las bellezas , una obra maestra de ...",El papá de las bellezas Felipe Trigo Realista ...
1,La última cerilla,Marie Vareille,Novela Psicológico Realista,,Las cifras no mienten: tres de cada cuatro niñ...,La última cerilla Marie Vareille Novela Psicol...
2,La enjuta,Víctor Català,Realista Relato,,En estos relatos la autora presenta una radiog...,La enjuta Víctor Català Realista Relato En est...
3,En el camping,Soledad Puértolas,Realista Relato,,El esplendor de Soledad Puértolas en su madure...,En el camping Soledad Puértolas Realista Relat...
4,Koljós,Emmanuel Carrère,Ensayo Memorias Novela Realista,,"Un libro total, una construcción narrativa des...",Koljós Emmanuel Carrère Ensayo Memorias Novela...


limpieza!!

In [56]:
df["puro"].head()[0]

'El papá de las bellezas Felipe Trigo Realista Relato El papá de las bellezas , una obra maestra de Felipe Trigo, despliega un enfoque audaz hacia las relaciones humanas y la complejidad de la belleza femenina. El libro, publicado en el contexto de un cambio social a inicios del siglo XX en España, refleja las tensiones y transformaciones de una sociedad en evolución. Con un estilo narrativo cuidadoso y evocador, Trigo sumerge al lector en una realidad donde la sensualidad se entrelaza con la crítica social, presentando personajes profundamente humanos que examinan sus propias motivaciones y deseos. Este entrelazado de lo bello y lo grotesco crea un ambiente vibrante que invita a la reflexión sobre el papel de la estética en la vida cotidiana.'

parte de WORD2VEC USANDO EL YA EXISTENTE


In [57]:

# 1. Cargar el modelo preentrenado
modelo = KeyedVectors.load_word2vec_format(
    "SBW-vectors-300-min5.bin",
    binary=True
)


In [58]:

# 2. Función para vectorizar cada libro
def vectorizar_texto(texto):
    if not isinstance(texto, str) or not texto.strip():
        return np.zeros(300, dtype=np.float32)

    palabras = texto.split()

    # Conservar únicamente las palabras conocidas por el modelo
    vectores = [
        modelo[palabra]
        for palabra in palabras
        if palabra in modelo
    ]

    # Si ninguna palabra existe en el vocabulario
    if not vectores:
        return np.zeros(300, dtype=np.float32)

    # Promediar los vectores de las palabras
    return np.mean(vectores, axis=0)


In [59]:
# Crear un DataFrame nuevo con las columnas id y vec_sbw
df_sbw = df[["id", "puro"]].copy()

df_sbw["vec_sbw"] = df_sbw["puro"].apply(vectorizar_texto)

# Conservar únicamente las columnas necesarias
df_sbw = df_sbw[["id", "vec_sbw"]]

# Verificar dimensiones del vector
print(df_sbw["vec_sbw"].iloc[0].shape)  # (300,)

# Mostrar las primeras filas
df_sbw.head()

(300,)


,id,vec_sbw
0,1,"[-0.015811114, -0.083094604, 0.067930214, -0.1..."
1,2,"[-0.04534435, -0.059153963, 0.07750422, -0.090..."
2,3,"[-0.013389647, -0.08637572, 0.091473505, -0.09..."
3,4,"[0.00068464753, -0.03950356, 0.0801799, -0.081..."
4,5,"[-0.035262242, -0.044648714, 0.041821692, -0.1..."


la idea es usar el vector para las busquedas y luego usar el id en el df de los libros 

In [60]:
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec


In [61]:
columnas_jugo = [
    "generos",
    "sinopsis"
]

df["jugo"] = df[columnas_jugo].agg(" ".join, axis=1)

df["jugo"] = (
    df["jugo"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [62]:
textos = df["jugo"].astype(str)


# Separar en oraciones y tokenizar
oraciones = []

for texto in textos:
    frases = re.split(r'[.!?¿¡\n]+', texto)

    for frase in frases:
        palabras = simple_preprocess(
            frase,
            deacc=True,  # Quita acentos
            min_len=2
        )

        if palabras:
            oraciones.append(palabras)


In [63]:

# Entrenar el modelo propio
modelo_propio = Word2Vec(
    sentences=oraciones,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    seed=42,
    epochs=10
)

print("Cantidad de oraciones:", len(oraciones))
print("Tamaño del vocabulario:", len(modelo_propio.wv))

Cantidad de oraciones: 732
Tamaño del vocabulario: 1679
